# DocChat: Agentic RAG Without the UI

This notebook turns the DocChat lab into runnable notebook cells. It uses the same core tools as the original project: Docling for document conversion, `MarkdownHeaderTextSplitter` for chunks, Chroma with OpenAI embeddings for vector search, LangChain's BM25 retriever, an ensemble retriever, OpenAI chat models for relevance/research/verification, and LangGraph for orchestration.

You need an `OPENAI_API_KEY` in the repository `.env` file or in your shell environment before running the indexing and agent cells.

Use `USE_OCR = True` to match the original DocChat/Docling OCR behavior. If your proxy blocks the RapidOCR model download, set `USE_OCR = False` in the configuration cell; table structure extraction remains enabled.

On Windows, `FORCE_HF_NO_SYMLINKS = True` avoids Hugging Face cache symlink errors such as `[WinError 1314] A required privilege is not held by the client`.

## 1. Imports and Configuration

In [3]:
import os
import shutil
from pathlib import Path
from typing import Literal
from typing_extensions import TypedDict

from dotenv import load_dotenv
from pydantic import BaseModel, Field

from langchain.tools import tool
from langchain_community.retrievers import BM25Retriever
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_classic.retrievers import EnsembleRetriever
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langgraph.graph import END, START, StateGraph

load_dotenv()

LAB_DIR = Path.cwd()
if LAB_DIR.name != "05_docchat":
    LAB_DIR = Path("02_Langchain_Langgraph/lab/05_docchat").resolve()

DOCCHAT_DIR = LAB_DIR / "docchat"
DOCUMENT_PATH = DOCCHAT_DIR / "examples" / "google-2024-environmental-report.pdf"
CHROMA_DIR = LAB_DIR / "chroma_docchat_notebook"
HF_CACHE_DIR = LAB_DIR / ".hf_cache"
HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("HF_HOME", str(HF_CACHE_DIR))
os.environ.setdefault("HF_HUB_CACHE", str(HF_CACHE_DIR / "hub"))
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")

CHAT_MODEL = os.getenv("OPENAI_CHAT_MODEL", "gpt-4.1-mini")
EMBEDDING_MODEL = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")
USE_OCR = False  # Set to False if RapidOCR model downloads are blocked by your proxy.
FORCE_HF_NO_SYMLINKS = True  # Windows without Developer Mode/admin cannot create HF cache symlinks.

if FORCE_HF_NO_SYMLINKS:
    import huggingface_hub.file_download as hf_file_download

    hf_file_download.are_symlinks_supported = lambda cache_dir=None: False

from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "OPENAI_API_KEY is required. Add it to the repository .env file "
        "or export it in your shell before running this notebook."
    )

print(f"Document: {DOCUMENT_PATH}")
print(f"Chat model: {CHAT_MODEL}")
print(f"Embedding model: {EMBEDDING_MODEL}")
print(f"Use OCR: {USE_OCR}")

Document: c:\Users\A200239740\git_repositories\agents_guide\02_Langchain_Langgraph\lab\05_docchat\docchat\examples\google-2024-environmental-report.pdf
Chat model: gpt-4.1-mini
Embedding model: text-embedding-3-small
Use OCR: False


## 2. Process a Real Document With Docling

The original DocChat app converts uploaded files with Docling, exports Markdown, and then splits by Markdown headers. Here we process the lab's Google environmental report, which contains tables and structured report sections.

In [4]:
def process_document(path: Path) -> list[Document]:
    if USE_OCR:
        converter = DocumentConverter()
    else:
        pipeline_options = PdfPipelineOptions()
        pipeline_options.do_ocr = False
        pipeline_options.do_table_structure = True
        converter = DocumentConverter(
            format_options={
                InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
            }
        )
    markdown = converter.convert(str(path)).document.export_to_markdown()

    splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=[("#", "Header 1"), ("##", "Header 2")]
    )
    chunks = splitter.split_text(markdown)

    for index, chunk in enumerate(chunks):
        chunk.metadata.update({"source": path.name, "chunk_id": index})

    return chunks


chunks = process_document(DOCUMENT_PATH)
print(f"Created {len(chunks)} chunks")
print(chunks[0].page_content[:1000])

Stage preprocess failed for run 1, pages [28]: std::bad_alloc
Stage preprocess failed for run 1, pages [29]: std::bad_alloc
Stage preprocess failed for run 1, pages [30]: std::bad_alloc
Stage preprocess failed for run 1, pages [31]: std::bad_alloc
Stage preprocess failed for run 1, pages [32]: std::bad_alloc
Stage preprocess failed for run 1, pages [33]: std::bad_alloc
Stage preprocess failed for run 1, pages [34]: std::bad_alloc
Stage preprocess failed for run 1, pages [35]: std::bad_alloc
Stage preprocess failed for run 1, pages [36]: std::bad_alloc
Stage preprocess failed for run 1, pages [37]: std::bad_alloc
Stage preprocess failed for run 1, pages [38]: std::bad_alloc
Stage preprocess failed for run 1, pages [39]: std::bad_alloc
Stage preprocess failed for run 1, pages [40]: std::bad_alloc
Stage preprocess failed for run 1, pages [41]: std::bad_alloc
Stage preprocess failed for run 1, pages [42]: std::bad_alloc
Stage preprocess failed for run 1, pages [48]: std::bad_alloc
Stage pr

Created 132 chunks
<!-- image -->


### Inspect Chunks That Look Table-Like

In [5]:
table_like_chunks = [doc for doc in chunks if "|" in doc.page_content or "PUE" in doc.page_content or "CFE" in doc.page_content]
print(f"Found {len(table_like_chunks)} table/metric-like chunks")
for doc in table_like_chunks[:3]:
    print("---")
    print(doc.metadata)
    print(doc.page_content[:1200])

Found 12 table/metric-like chunks
---
{'Header 2': 'Table of contents', 'source': 'google-2024-environmental-report.pdf', 'chunk_id': 4}
| Introduction                 |   2 | Appendix                     |   60 |
|------------------------------|-----|------------------------------|------|
| Executive letter             |   3 | About Google                 |   61 |
| Our sustainability strategy  |   5 | Sustainability governance    |   61 |
| 2023 highlights              |   6 | Risk management              |   61 |
| Targets and progress summary |   7 | Stakeholder engagement       |   62 |
| Searching for sustainability |   8 | and partnership              |      |
|                              |     | Multi-sector products        |   67 |
| AI for sustainability        |   9 | Ecosystems for collaboration |   68 |
| Our products                 |  14 | Environmental data           |   70 |
| Mitigation                   |  16 | Certifications               |   80 |
| Adaptation and

## 3. Build Hybrid Retrieval: Chroma + BM25

DocChat combines semantic vector search and keyword search. Chroma stores OpenAI embedding vectors; BM25 catches exact terms such as table labels, years, metric names, and facility names.

In [6]:
REBUILD_CHROMA = True

if REBUILD_CHROMA and CHROMA_DIR.exists():
    shutil.rmtree(CHROMA_DIR)

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=str(CHROMA_DIR),
    collection_name="docchat_google_environmental_report",
)
vector_retriever = vector_store.as_retriever(search_kwargs={"k": 8})

bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 8

hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.4, 0.6],
)

print("Hybrid retriever ready")

Hybrid retriever ready


### Retrieval Tool Shape

The graph below calls the retriever directly, but this is the same retrieval function exposed as a LangChain tool. This shape is useful when turning the workflow into a tool-calling agent.

In [7]:
@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """Retrieve document chunks that may answer the user question."""
    docs = hybrid_retriever.invoke(query)
    serialized = "\n\n".join(
        f"Source: {doc.metadata}\nContent: {doc.page_content}"
        for doc in docs
    )
    return serialized, docs


sample_question = (
    "Retrieve the data center PUE efficiency values in Singapore 2nd facility "
    "in 2019 and 2022. Also retrieve regional average CFE in Asia Pacific in 2023."
)

retrieved_docs = hybrid_retriever.invoke(sample_question)
for index, doc in enumerate(retrieved_docs[:3], start=1):
    print(f"--- retrieved chunk {index} ---")
    print(doc.metadata)
    print(doc.page_content[:1200])

--- retrieved chunk 1 ---
{'Header 2': 'Endnotes', 'source': 'google-2024-environmental-report.pdf', 'chunk_id': 130}
- 1 This calculation is based on internal data, as of May 2024.
- 2 'The Carbon Footprint of Machine Learning Training Will Plateau, Then Shrink,' Computer, vol. 55, July 2022.
- 3 According to Google's own analysis of our more efficient servers, power infrastructure, and cooling systems, compared with data center industry averages based on 2023 data. Uptime Institute's annual data center survey from 2023 noted that the primary contributor to the flatlining of the industry average PUE is a richer geographical mix of surveyed data centers, with an increasing number of data centers in the Asia, Middle East, Africa, and Latin America regions. Facilities in these regions tend to be smaller in capacity and located in warmer climates-both factors which typically require greater energy consumption.
- 4 According to the Uptime Institute's 2023 Global Data Center Survey, the glo

## 4. Define the Agents

The original repo has separate relevance, research, and verification classes. In the notebook, each role is a LangGraph node backed by OpenAI chat calls.

In [8]:
llm = ChatOpenAI(model=CHAT_MODEL, temperature=0)


class RelevanceDecision(BaseModel):
    label: Literal["CAN_ANSWER", "PARTIAL", "NO_MATCH"] = Field(
        description="Whether the retrieved document context can answer the question."
    )
    reason: str = Field(description="Brief reason for the classification.")


class VerificationDecision(BaseModel):
    supported: Literal["YES", "NO"]
    unsupported_claims: list[str]
    contradictions: list[str]
    relevant: Literal["YES", "NO"]
    additional_details: str


relevance_llm = llm.with_structured_output(RelevanceDecision)
verification_llm = llm.with_structured_output(VerificationDecision)


def serialize_docs(docs: list[Document], max_chars: int = 12000) -> str:
    serialized = []
    total = 0
    for doc in docs:
        text = f"Source: {doc.metadata}\n{doc.page_content}"
        if total + len(text) > max_chars:
            break
        serialized.append(text)
        total += len(text)
    return "\n\n---\n\n".join(serialized)


class DocChatState(TypedDict):
    question: str
    documents: list[Document]
    relevance: Literal["CAN_ANSWER", "PARTIAL", "NO_MATCH"]
    relevance_reason: str
    draft_answer: str
    verification_report: str
    retry_count: int


def retrieve_node(state: DocChatState) -> dict:
    docs = hybrid_retriever.invoke(state["question"])
    return {"documents": docs}


def check_relevance_node(state: DocChatState) -> dict:
    context = serialize_docs(state["documents"], max_chars=8000)
    decision = relevance_llm.invoke(
        [
            {
                "role": "system",
                "content": (
                    "You are a relevance checker for a document QA system. "
                    "Classify whether the retrieved context can answer the question. "
                    "Use CAN_ANSWER for enough explicit support, PARTIAL for related but incomplete support, "
                    "and NO_MATCH when the context is unrelated."
                ),
            },
            {
                "role": "user",
                "content": f"Question:\n{state['question']}\n\nRetrieved context:\n{context}",
            },
        ]
    )
    update = {"relevance": decision.label, "relevance_reason": decision.reason}
    if decision.label == "NO_MATCH":
        update["draft_answer"] = "This question is not related to the uploaded document context."
    return update


def research_node(state: DocChatState) -> dict:
    context = serialize_docs(state["documents"], max_chars=14000)
    response = llm.invoke(
        [
            {
                "role": "system",
                "content": (
                    "You are the DocChat research agent. Answer only from the provided context. "
                    "Preserve numeric values, years, table labels, and units. "
                    "If the context is insufficient, say exactly what is missing."
                ),
            },
            {
                "role": "user",
                "content": f"Question:\n{state['question']}\n\nContext:\n{context}",
            },
        ]
    )
    return {"draft_answer": response.content}


def verify_node(state: DocChatState) -> dict:
    context = serialize_docs(state["documents"], max_chars=14000)
    decision = verification_llm.invoke(
        [
            {
                "role": "system",
                "content": (
                    "You are the DocChat verification agent. Check the answer against the context. "
                    "Flag unsupported claims, contradictions, and irrelevant answers."
                ),
            },
            {
                "role": "user",
                "content": (
                    f"Question:\n{state['question']}\n\n"
                    f"Answer:\n{state['draft_answer']}\n\n"
                    f"Context:\n{context}"
                ),
            },
        ]
    )
    report = (
        f"Supported: {decision.supported}\n"
        f"Unsupported Claims: {decision.unsupported_claims or 'None'}\n"
        f"Contradictions: {decision.contradictions or 'None'}\n"
        f"Relevant: {decision.relevant}\n"
        f"Additional Details: {decision.additional_details}"
    )
    retry_increment = 1 if decision.supported == "NO" or decision.relevant == "NO" else 0
    return {
        "verification_report": report,
        "retry_count": state["retry_count"] + retry_increment,
    }


## 5. Build the LangGraph Workflow

In [9]:
def route_after_relevance(state: DocChatState) -> Literal["research", "__end__"]:
    return "research" if state["relevance"] in {"CAN_ANSWER", "PARTIAL"} else END


def route_after_verification(state: DocChatState) -> Literal["research", "__end__"]:
    report = state["verification_report"].lower()
    failed = "supported: no" in report or "relevant: no" in report
    if failed and state["retry_count"] < 2:
        return "research"
    return END


builder = StateGraph(DocChatState)
builder.add_node("retrieve", retrieve_node)
builder.add_node("check_relevance", check_relevance_node)
builder.add_node("research", research_node)
builder.add_node("verify", verify_node)

builder.add_edge(START, "retrieve")
builder.add_edge("retrieve", "check_relevance")
builder.add_conditional_edges("check_relevance", route_after_relevance)
builder.add_edge("research", "verify")
builder.add_conditional_edges("verify", route_after_verification)

docchat_graph = builder.compile()
print("Graph compiled")

Graph compiled


## 6. Run the Agentic RAG Pipeline

In [10]:
def ask_docchat(question: str) -> dict:
    return docchat_graph.invoke(
        {
            "question": question,
            "documents": [],
            "relevance": "NO_MATCH",
            "relevance_reason": "",
            "draft_answer": "",
            "verification_report": "",
            "retry_count": 0,
        }
    )


result = ask_docchat(sample_question)

print("Relevance:", result["relevance"])
print("Reason:", result["relevance_reason"])
print("\nAnswer:\n", result["draft_answer"])
print("\nVerification:\n", result["verification_report"])
print("\nRetrieved sources:")
for doc in result["documents"][:5]:
    print(doc.metadata)

Relevance: NO_MATCH
Reason: The retrieved context is empty and does not contain any information about data center PUE efficiency values or regional average CFE.

Answer:
 This question is not related to the uploaded document context.

Verification:
 

Retrieved sources:
{'Header 2': 'Endnotes', 'source': 'google-2024-environmental-report.pdf', 'chunk_id': 130}
{'Header 2': 'Alternative water sources', 'source': 'google-2024-environmental-report.pdf', 'chunk_id': 115}
{'Header 2': "Contextualizing Google's impact", 'source': 'google-2024-environmental-report.pdf', 'chunk_id': 41}
{'chunk_id': 19, 'source': 'google-2024-environmental-report.pdf', 'Header 2': 'Achieved at least 90% carbon-free energy in 10 grid regions'}
{'Header 2': 'Energy efficiency at Google data centers', 'source': 'google-2024-environmental-report.pdf', 'chunk_id': 47}


### Out-of-Scope Check

In [ ]:
irrelevant = ask_docchat("Who won the 2015 World Series?")
print("Relevance:", irrelevant["relevance"])
print("Reason:", irrelevant["relevance_reason"])
print("Answer:", irrelevant["draft_answer"])